## Learning notes — 03.ABAC Policies with Governed Tags

### Learning objectives
Follow **03.ABAC Policies with Governed Tags** from its source inputs through its operations to the archived results.
- Explain schema, security, write, profiling as used in this lesson.

### Source-specific learning focus
- Compare raw fields with typed records and parsing failures.
- Explain how identity and governance affect visible data and permitted actions.
- Trace how source records change persistent destination state.
- Interpret selected fields, samples and counts at their actual output grain.

### Input/output interpretation
Original source and outputs are preserved unchanged and were not executed. Saved tables, HTML displays and errors are archived generated outputs, not supplied datasets. Inline literals are examples already present in the source. Cloud storage, remote tables, catalogs, services and checkpoints are unavailable locally unless a supplied snapshot is explicitly listed below; no external input is claimed available.

SQL relation identifiers referenced in source (inputs and/or destinations; not a claim that these objects exist locally): `customers_abac`, `sales_abac`.


# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

# Data-Level Security Using ABAC (Governed Tags)
This notebook demonstrates how data-level security can be implemented using Attribute-Based Access Control (ABAC) in Unity Catalog.

In previous examples, row filters and column masks were applied directly to individual tables. While effective, that approach requires policies to be defined and managed separately for each table.

ABAC addresses this by introducing a centralized model where:

- Data is tagged using governed attributes
- Policies are defined once based on those attributes
- Policies are applied automatically across multiple objects

The objective is to:

- Classify sensitive data using governed tags
- Define reusable policies based on those tags

> **Note:** Row filters and column masks are not supported on Dedicated (Single User) clusters as of now.
> Use a Shared cluster or a Serverless cluster to run this demo.

#### Step 1: Set up the schema for the table

## Learning notes — before source operation

**schema**: Compare declared or inferred field types with raw input. Parsing, schema evolution and rescued/corrupt records can produce nulls or additional columns rather than clean typed rows.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Parsing a numeric field from an incompatible string can fail or yield null depending on parsing mode; inspect the declared schema and error policy.

Archived Databricks source — shown for reading, not Python execution.

```sql
USE CATALOG demo;
USE SCHEMA data_security;
```

In [ ]:
# Archived output only; no computation.

Archived MIME payload:
{
  "application/vnd.databricks.empty-table+json": {
    "directive_name": "NoDirective"
  }
}

#### Step 2: Create the table
A table is defined to represent sales data across multiple regions.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE TABLE sales_abac (
  id INT,
  region STRING,
  email STRING,
  revenue INT
);

```

In [ ]:
# Archived output only; no computation.

Archived MIME payload:
{
  "application/vnd.databricks.empty-table+json": {
    "directive_name": "NoDirective"
  }
}

#### Step 3: Insert Sample Data

## Learning notes — before source operation

**write**: This operation changes persistent state. Identify append, overwrite or incremental-copy semantics before rerunning; archived output does not establish the current destination contents.

Archived Databricks source — shown for reading, not Python execution.

```sql
INSERT INTO sales_abac VALUES
  (1, 'UK', 'john.doe@email.com', 10000),
  (2, 'US', 'mike.smith@email.com', 20000),
  (3, 'UK', 'sara.jones@email.com', 15000),
  (4, 'US', 'david.brown@email.com', 25000),
  (5, 'UK', 'emma.wilson@email.com', 18000);
```

In [ ]:
# Archived output only; no computation.

Archived table:
num_affected_rows	num_inserted_rows
5	5

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM sales_abac;
```

In [ ]:
# Archived output only; no computation.

Archived table:
id	region	email	revenue
1	UK	john.doe@email.com	10000
2	US	mike.smith@email.com	20000
3	UK	sara.jones@email.com	15000
4	US	david.brown@email.com	25000
5	UK	emma.wilson@email.com	18000

#### Step 4: Create UDFs for row filter and column mask  
(We have already created them as part of previous lecture)
1. `fn_filter_region` (This function evaluates the user’s group membership and determines whether to include a specifc record in the output based on the region)
1. `fn_mask_email` (This function evaluates the user’s group membership and determines whether to return the original value or a masked version of the value)

#### Step 5: Create Governed Tags
1. `pii = email` (Indicates that the data contains personally identifiable information (PII))
1. `access_type = region` (Defines the type of access control to be applied to the data, such as region-based or department-based filtering)

#### Step 6: Create Policy 
(Could be at catalog/ schema/ table level)
1. policy_filter_region
1. policy_mask_email

#### Step 7: Attach tags to the table
1. sales_abac.region
1. sales_abac.email

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM sales_abac;
```

In [ ]:
# Archived output only; no computation.

Archived table:
id	region	email	revenue
1	UK	j***@email.com	10000
3	UK	s***@email.com	15000
5	UK	e***@email.com	18000

#### Step 8: Create another table to demonstrate scalability of the ABAC implementation

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE TABLE customers_abac (
  customer_id INT,
  customer_name STRING,
  email STRING,
  region STRING
);
```

In [ ]:
# Archived output only; no computation.

Archived MIME payload:
{
  "application/vnd.databricks.empty-table+json": {
    "directive_name": "NoDirective"
  }
}

Archived Databricks source — shown for reading, not Python execution.

```sql
ALTER TABLE customers_abac
ALTER COLUMN email
SET TAGS ('pii' = 'email');
```

In [ ]:
# Archived output only; no computation.

Archived MIME payload:
{
  "application/vnd.databricks.empty-table+json": {
    "directive_name": "NoDirective"
  }
}

Archived Databricks source — shown for reading, not Python execution.

```sql
ALTER TABLE customers_abac
ALTER COLUMN region
SET TAGS ('access_type' = 'region');
```

In [ ]:
# Archived output only; no computation.

Archived MIME payload:
{
  "application/vnd.databricks.empty-table+json": {
    "directive_name": "NoDirective"
  }
}

## Learning notes — before source operation

**write**: This operation changes persistent state. Identify append, overwrite or incremental-copy semantics before rerunning; archived output does not establish the current destination contents.

Archived Databricks source — shown for reading, not Python execution.

```sql
INSERT INTO customers_abac VALUES
  (1, 'John Smith', 'john.smith@email.com', 'UK'),
  (2, 'Mike Johnson', 'mike.johnson@email.com', 'US'),
  (3, 'Sara Jones', 'sara.jones@email.com', 'UK'),
  (4, 'David Brown', 'david.brown@email.com', 'US');
```

In [ ]:
# Archived output only; no computation.

Archived table:
num_affected_rows	num_inserted_rows
4	4

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM customers_abac;
```

In [ ]:
# Archived output only; no computation.

Archived table:
customer_id	customer_name	email	region
1	John Smith	j***@email.com	UK
3	Sara Jones	s***@email.com	UK